In [4]:
from sysdata.sim.csv_futures_sim_data import csvFuturesSimData

data = csvFuturesSimData()


In [3]:
data.get_instrument_list()


['FTSECHINAH',
 'EU-MID',
 'EU-DIV30',
 'JP-REALESTATE',
 'SMI',
 'DJSTX-SMALL',
 'LUMBER-new',
 'EUROSTX200-LARGE',
 'CAD5',
 'PALLAD',
 'HIGHYIELD',
 'GBPEUR',
 'SP500_micro',
 'BOVESPA',
 'IRON',
 'LEANHOG',
 'GAS-LAST',
 'US-STAPLES',
 'ETHER-micro',
 'WHEAT_ICE',
 'VNKI',
 'SOYBEAN',
 'EU-DJ-UTIL',
 'FTSEINDO',
 'NZD',
 'ETHANOL',
 'GOLD',
 'MSCIASIA',
 'BB3M',
 'CAD2',
 'EU-DJ-TECH',
 'JPY_mini',
 'EU-TECH',
 'MIB',
 'SMI-MID',
 'GILT',
 'BTP',
 'EU-TRAVEL',
 'MSCIWORLD',
 'SWISSLEAD',
 'SOYBEAN_mini',
 'KRWUSD_mini',
 'STEEL',
 'HANGTECH',
 'NASDAQ_micro',
 'ROBUSTA',
 'VIX_mini',
 'CAD10',
 'AEX',
 'BBCOMM',
 'NASDAQ',
 'EU-HOUSE',
 'HOUSE-US',
 'NIKKEI400',
 'RUSSELL',
 'GASOILINE_micro',
 'EURIBOR-ICE',
 'CANOLA',
 'EU-HEALTH',
 'HEATOIL-ICE',
 'MSCIEAFA',
 'SUGAR16',
 'SILVER',
 'EUA',
 'COPPER-micro',
 'CHEESE',
 'CAD_micro',
 'EURCHF',
 'GAS-PEN',
 'MILKWET',
 'CAC',
 'CHF_micro',
 'GASOIL',
 'AUD',
 'EU-CHEM',
 'EU-DJ-OIL',
 'BUND',
 'EU-FOOD',
 'US-MATERIAL',
 'US10',
 '

In [ ]:
data.keys()
data["SP500_micro"]


index
1982-09-14 23:00:00     679.40
1982-09-15 23:00:00     679.90
1982-09-16 23:00:00     679.25
1982-09-17 23:00:00     678.35
1982-09-20 23:00:00     679.15
                        ...   
2024-03-28 17:00:00    5309.25
2024-03-28 18:00:00    5316.50
2024-03-28 19:00:00    5305.50
2024-03-28 20:00:00    5306.00
2024-03-28 23:00:00    5303.75
Name: price, Length: 35898, dtype: float64

In [ ]:
import pandas as pd
from sysquant.estimators.vol import robust_vol_calc


def calc_ewmac_forecast(price, Lfast, Lslow=None):
    price = price.resample("1B").last()
    if Lslow is None:
        Lslow = 4 * Lfast

    fast_ewma = price.ewm(span=Lfast).mean()
    slow_ewma = price.ewm(span=Lslow).mean()
    raw_ewmac = fast_ewma - slow_ewma

    vol = robust_vol_calc(price.diff())

    return raw_ewmac / vol


In [ ]:
instrument_code = "SP500_micro"
price = data.daily_prices(instrument_code)
ewmac = calc_ewmac_forecast(price, 32, 128)
ewmac.tail(5)


index
2024-03-22    8.367249
2024-03-25    8.575964
2024-03-26    8.771066
2024-03-27    8.856622
2024-03-28    9.108725
Freq: B, Name: price, dtype: float64

In [5]:
from matplotlib.pyplot import show

ewmac.plot()
show()


NameError: name 'ewmac' is not defined

In [27]:
from sysdata.sim.csv_futures_sim_data import csvFuturesSimData

data = csvFuturesSimData()

from systems.provided.rules.ewmac import ewmac_forecast_with_defaults as ewmac

from systems.forecasting import Rules

my_rules = Rules(ewmac)
my_rules.trading_rules()


{'rule0': TradingRule; function: <function ewmac_forecast_with_defaults at 0x121ead480>, data: data.daily_prices (args: {}) and other_args: }

In [28]:
type(data)


sysdata.sim.csv_futures_sim_data.csvFuturesSimData

In [29]:
my_rules = Rules(dict(ewmac=ewmac))

my_rules.trading_rules()


{'ewmac': TradingRule; function: <function ewmac_forecast_with_defaults at 0x121ead480>, data: data.daily_prices (args: {}) and other_args: }

In [30]:
from systems.basesystem import System

my_system = System([my_rules], data)
my_system


Private configuration '/Users/jasonli/Dev/FORKed repo/pysystemtrade/private/private_config.yaml' is missing or misconfigured; no problem if running in sim mode


System base_system with .config, .data, and .stages: rules

In [ ]:
my_system.rules.get_raw_forecast("SOFR", "ewmac").tail(5)


2026-05-21 14:50:11 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-05-21 14:50:11 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-05-21 14:50:11 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-05-21 14:50:11 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SOFR'} Calculating raw forecast SOFR for ewmac


index
2024-03-22   -0.349853
2024-03-25   -0.378965
2024-03-26   -0.394422
2024-03-27   -0.381021
2024-03-28   -0.378662
Freq: B, Name: price, dtype: float64

In [ ]:
from systems.trading_rules import TradingRule

ewmac_rule = TradingRule(ewmac)
my_rules = Rules(dict(ewmac=ewmac_rule))
ewmac_rule


TradingRule; function: <function ewmac_forecast_with_defaults at 0x121ead480>, data: data.daily_prices (args: {}) and other_args: 

In [ ]:
ewmac_8 = TradingRule(
    (ewmac, [], dict(Lfast=8, Lslow=32))
)  ## as a tuple (function, data, other_args) notice the empty element in the middle
ewmac_32 = TradingRule(
    dict(function=ewmac, other_args=dict(Lfast=32, Lslow=128))
)  ## as a dict
my_rules = Rules(dict(ewmac8=ewmac_8, ewmac32=ewmac_32))
my_rules.trading_rules()["ewmac32"]


TradingRule; function: <function ewmac_forecast_with_defaults at 0x121ead480>, data: data.daily_prices (args: {}) and other_args: Lfast, Lslow

In [ ]:
my_system = System([my_rules], data)
my_system.rules.get_raw_forecast("SOFR", "ewmac32").tail(5)


Private configuration '/Users/jasonli/Dev/FORKed repo/pysystemtrade/private/private_config.yaml' is missing or misconfigured; no problem if running in sim mode
2026-05-21 15:02:23 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-05-21 15:02:23 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-05-21 15:02:23 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-05-21 15:02:23 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SOFR'} Calculating raw forecast SOFR for ewmac32


index
2024-03-22   -0.349853
2024-03-25   -0.378965
2024-03-26   -0.394422
2024-03-27   -0.381021
2024-03-28   -0.378662
Freq: B, Name: price, dtype: float64

In [ ]:
from sysdata.config.configdata import Config

my_config = Config()
my_config


Config with elements: 

In [38]:
empty_rules = Rules()
my_config.trading_rules = dict(ewmac8=ewmac_8, ewmac32=ewmac_32)
my_system = System([empty_rules], data, my_config)

my_system.rules.get_raw_forecast("SOFR", "ewmac8")


Private configuration '/Users/jasonli/Dev/FORKed repo/pysystemtrade/private/private_config.yaml' is missing or misconfigured; no problem if running in sim mode
2026-05-21 15:12:45 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-05-21 15:12:45 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-05-21 15:12:45 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-05-21 15:12:45 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SOFR'} Calculating raw forecast SOFR for ewmac8


index
1984-03-23         NaN
1984-03-26         NaN
1984-03-27         NaN
1984-03-28         NaN
1984-03-29         NaN
                ...   
2024-03-22   -0.792683
2024-03-25   -0.737074
2024-03-26   -0.648340
2024-03-27   -0.476329
2024-03-28   -0.381107
Freq: B, Name: price, Length: 10440, dtype: float64

In [ ]:
from systems.forecast_scale_cap import ForecastScaleCap

my_config.instruments = ["SOFR", "US10", "CORN", "SP500_micro"]
my_config.use_forecast_scale_estimates = True

fcs = ForecastScaleCap()
my_system = System([fcs, my_rules], data, my_config)

my_system.forecastScaleCap.get_forecast_scalar("SOFR", "ewmac32").tail(5)


Private configuration '/Users/jasonli/Dev/FORKed repo/pysystemtrade/private/private_config.yaml' is missing or misconfigured; no problem if running in sim mode
2026-05-21 15:15:18 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-05-21 15:15:18 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-05-21 15:15:18 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-05-21 15:15:18 DEBUG base_system {'stage': 'forecastScaleCap'} Getting cross sectional forecasts for scalar calculation for ewmac32 over CORN, SOFR, SP500_micro, US10
2026-05-21 15:15:18 DEBUG base_system {'stage': 'rules', 'instrument_code': 'CORN'} Calculating raw forecast CORN for ewmac32
2026-05-21 15:15:19 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SOFR'} Calculating raw forecast SOFR for ewmac32
2026-05-21 15:15:19 DEBUG base_system {'stage': '

index
2024-03-22    2.975027
2024-03-25    2.975012
2024-03-26    2.975015
2024-03-27    2.975025
2024-03-28    2.975090
Freq: B, dtype: float64

In [ ]:
my_config.forecast_scalars = dict(ewmac8=5.3, ewmac32=2.65)
my_config.use_forecast_scale_estimates = False

my_system = System([fcs, my_rules], data, my_config)

my_system.forecastScaleCap.get_forecast_scalar("SOFR", "ewmac32").tail(5)


Private configuration '/Users/jasonli/Dev/FORKed repo/pysystemtrade/private/private_config.yaml' is missing or misconfigured; no problem if running in sim mode
2026-05-21 15:16:42 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-05-21 15:16:42 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-05-21 15:16:42 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-05-21 15:16:42 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SOFR'} Calculating raw forecast SOFR for ewmac32


index
2024-03-22    2.65
2024-03-25    2.65
2024-03-26    2.65
2024-03-27    2.65
2024-03-28    2.65
Freq: B, dtype: float64

In [ ]:
my_system.forecastScaleCap.get_capped_forecast("SOFR", "ewmac32")


2026-05-21 15:17:00 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'SOFR'} Calculating capped forecast for SOFR ewmac32


index
1984-03-23         NaN
1984-03-26         NaN
1984-03-27         NaN
1984-03-28         NaN
1984-03-29         NaN
                ...   
2024-03-22   -0.927110
2024-03-25   -1.004257
2024-03-26   -1.045217
2024-03-27   -1.009705
2024-03-28   -1.003454
Freq: B, Length: 10440, dtype: float64

In [43]:
from systems.forecast_combine import ForecastCombine

combiner = ForecastCombine()
my_system = System([fcs, empty_rules, combiner], data, my_config)

my_system.combForecast.get_forecast_weights("SOFR").tail(5)


Private configuration '/Users/jasonli/Dev/FORKed repo/pysystemtrade/private/private_config.yaml' is missing or misconfigured; no problem if running in sim mode
2026-05-21 15:19:51 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-05-21 15:19:51 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-05-21 15:19:51 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-05-21 15:19:51 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Calculating forecast weights for SOFR
2026-05-21 15:19:51 WARNING base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} WARNING: No forecast weights  - using equal weights of 0.500 over all 2 trading rules in system
2026-05-21 15:19:51 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'SOFR'} Calculating capped forecast for SOFR ewmac32
2026-05-21 15:1

,ewmac8,ewmac32
index,,
2024-03-22,0.5,0.5
2024-03-25,0.5,0.5
2024-03-26,0.5,0.5
2024-03-27,0.5,0.5
2024-03-28,0.5,0.5


In [44]:
my_system.combForecast.get_forecast_diversification_multiplier("SOFR").tail(5)


2026-05-21 15:19:56 INFO base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Calculating forecast div multiplier for SOFR
2026-05-21 15:19:56 INFO base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Calculating forecast correlations over CORN, SOFR, SP500_micro, US10
2026-05-21 15:19:56 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'CORN'} Calculating capped forecast for CORN ewmac32
2026-05-21 15:19:56 DEBUG base_system {'stage': 'rules', 'instrument_code': 'CORN'} Calculating raw forecast CORN for ewmac32
2026-05-21 15:19:56 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'CORN'} Calculating capped forecast for CORN ewmac8
2026-05-21 15:19:56 DEBUG base_system {'stage': 'rules', 'instrument_code': 'CORN'} Calculating raw forecast CORN for ewmac8
2026-05-21 15:19:56 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'SP500_micro'} Calculating capped forecast for SP500_micro ewmac32
2026-05-21 15:19:56 DE

index
2024-03-22    1.106181
2024-03-25    1.106183
2024-03-26    1.106185
2024-03-27    1.106187
2024-03-28    1.106189
Freq: B, dtype: float64

In [ ]:
from systems.rawdata import RawData
from systems.positionsizing import PositionSizing
from systems.accounts.accounts_stage import Account

combiner = ForecastCombine()
raw_data = RawData()
position_size = PositionSizing()
my_account = Account()

## let's use naive markowitz to get more interesting results...
my_config.forecast_weight_estimate = dict(method="one_period")
my_config.use_forecast_weight_estimates = True
my_config.use_forecast_div_mult_estimates = True

combiner = ForecastCombine()
my_system = System(
    [my_account, fcs, my_rules, combiner, position_size, raw_data], data, my_config
)

print(my_system.combForecast.get_forecast_weights("US10").tail(5))
print(my_system.combForecast.get_forecast_diversification_multiplier("US10").tail(5))


In [ ]:
from systems.provided.futures_chapter15.basesystem import futures_system

system = futures_system()
system.portfolio.get_notional_position("SOFR")


Private configuration '/Users/jasonli/Dev/FORKed repo/pysystemtrade/private/private_config.yaml' is missing or misconfigured; no problem if running in sim mode
2026-05-25 20:58:14 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-05-25 20:58:14 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-05-25 20:58:14 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-05-25 20:58:14 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'SOFR'} Calculating notional position for SOFR
2026-05-25 20:58:14 INFO base_system {'stage': 'portfolio', 'instrument_code': 'SOFR'} Calculating instrument weights
2026-05-25 20:58:14 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'SOFR'} Calculating raw instrument weights
2026-05-25 20:58:14 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'ba

index
1984-03-23         NaN
1984-03-26         NaN
1984-03-27         NaN
1984-03-28         NaN
1984-03-29         NaN
                ...   
2024-03-22   -0.525723
2024-03-25   -0.548660
2024-03-26   -0.569837
2024-03-27   -0.545828
2024-03-28   -0.553313
Freq: B, Length: 10440, dtype: float64

In [ ]:
system.config.trading_rules.keys()


dict_keys(['ewmac2_8', 'ewmac4_16', 'ewmac8_32', 'ewmac16_64', 'ewmac32_128', 'ewmac64_256', 'carry'])

In [ ]:
system.accounts.methods()


['average_forecast',
 'capital_multiplier',
 'config',
 'forecast_cap',
 'forecast_diversification_multiplier',
 'forecast_turnover',
 'forecast_weight',
 'forecast_weights_for_instrument',
 'get_SR_cost_for_instrument_forecast',
 'get_SR_cost_given_turnover',
 'get_SR_cost_per_trade_for_instrument',
 'get_SR_holding_cost_only',
 'get_SR_trading_cost_only_given_turnover',
 'get_SR_transaction_cost_for_instrument_forecast',
 'get_actual_buffers_for_position',
 'get_actual_capital',
 'get_actual_position',
 'get_annual_risk_target',
 'get_average_position_at_subsystem_level',
 'get_average_position_for_instrument_at_portfolio_level',
 'get_buffered_position',
 'get_buffered_position_with_multiplier',
 'get_buffered_subsystem_position',
 'get_buffers_for_position',
 'get_buffers_for_subsystem_position',
 'get_capped_forecast',
 'get_daily_percentage_volatility',
 'get_daily_prices',
 'get_daily_returns_volatility',
 'get_fx_rate',
 'get_hourly_prices',
 'get_instrument_diversification_mul

In [25]:
system.rules


Rules object with rules ewmac2_8, ewmac4_16, ewmac8_32, ewmac16_64, ewmac32_128, ewmac64_256, carry

In [ ]:
from sysdata.config.configdata import Config

my_config = Config("private.intro_system.testconfig.yaml")
system2 = futures_system(config=my_config)


Private configuration '/Users/jasonli/Dev/FORKed repo/pysystemtrade/private/private_config.yaml' is missing or misconfigured; no problem if running in sim mode


In [ ]:
system2.rules


Rules object with rules ewmac8_32, ewmac16_64, ewmac32_128, ewmac64_256, carry

In [31]:
from sysdata.sim.csv_futures_sim_data import csvFuturesSimData
from sysdata.config.configdata import Config
from systems.forecasting import Rules
from systems.basesystem import System
from systems.forecast_combine import ForecastCombine
from systems.forecast_scale_cap import ForecastScaleCap
from systems.rawdata import RawData
from systems.positionsizing import PositionSizing
from systems.portfolio import Portfolios
from systems.accounts.accounts_stage import Account

data = csvFuturesSimData()
config = Config("private.intro_system.c003.yaml")

system = System(
    [
        Account(),
        Portfolios(),
        PositionSizing(),
        RawData(),
        ForecastCombine(),
        ForecastScaleCap(),
        Rules(),
    ],
    data,
    config,
)

system.rawdata.daily_returns("SP500_micro")


Private configuration '/Users/jasonli/Dev/FORKed repo/pysystemtrade/private/private_config.yaml' is missing or misconfigured; no problem if running in sim mode
2026-05-27 12:32:46 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-05-27 12:32:46 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-05-27 12:32:46 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']


2026-05-27 12:32:46 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'SP500_micro'} Calculating daily prices for SP500_micro


index
1982-09-14      NaN
1982-09-15     0.50
1982-09-16    -0.65
1982-09-17    -0.90
1982-09-20     0.80
              ...  
2024-03-22   -18.00
2024-03-25    -8.50
2024-03-26    -8.50
2024-03-27    35.75
2024-03-28    -4.25
Freq: B, Name: price, Length: 10838, dtype: float64

In [ ]:
from sysdata.config.configdata import Config
from systems.provided.futures_chapter15.basesystem import futures_system

my_config = Config("private.intro_system.c003.yaml")
system = futures_system(config=my_config)


Private configuration '/Users/jasonli/Dev/FORKed repo/pysystemtrade/private/private_config.yaml' is missing or misconfigured; no problem if running in sim mode


In [ ]:
system.config


Config with elements: GMT_offset_hours, average_absolute_forecast, backtest_compress, backtest_max_age, backtest_store_directory, base_currency, broker_factory_func, buffer_method, buffer_size, buffer_trade_to_edge, capital_multiplier, csv_backup_directory, duplicate_instruments, echo_directory, echo_extension, exclude_instrument_lists, execution_algos, forecast_cap, forecast_correlation_estimate, forecast_cost_estimates, forecast_div_mult_estimate, forecast_div_multiplier, forecast_post_ceiling_cost_SR, forecast_scalar, forecast_scalar_estimate, forecast_weight_estimate, forecast_weight_ewma_span, forecast_weights, ib_idoffset, ib_ipaddress, ib_port, ignore_future_prices, ignore_negative_prices, ignore_prices_with_zero_volumes_daily, ignore_prices_with_zero_volumes_intraday, ignore_zero_prices, instrument_correlation_estimate, instrument_div_mult_estimate, instrument_div_multiplier, instrument_returns_correlation, instrument_weight_estimate, instrument_weight_ewma_span, instrument_wei

In [33]:
system.accounts.portfolio().percent.stats()


2026-05-27 12:33:13 INFO base_system {'stage': 'accounts'} Calculating pandl for portfolio
2026-05-27 12:33:13 DEBUG base_system {'stage': 'positionSize'} Getting vol target
2026-05-27 12:33:13 DEBUG base_system {'stage': 'accounts', 'instrument_code': 'SP500'} Calculating pandl for instrument for SP500
2026-05-27 12:33:13 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'SP500'} Calculating notional position for SP500
2026-05-27 12:33:13 INFO base_system {'stage': 'portfolio', 'instrument_code': 'SP500'} Calculating instrument weights
2026-05-27 12:33:13 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'SP500'} Calculating raw instrument weights
2026-05-27 12:33:13 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-05-27 12:33:13 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-05-27 12:33:13 DEBUG base_system Following instruments are marked as 'bad_market

[[('min', '-20.46'),
  ('max', '8.94'),
  ('median', '0.07062'),
  ('mean', '0.06183'),
  ('std', '1.381'),
  ('skew', '-0.5568'),
  ('ann_mean', '15.83'),
  ('ann_std', '22.1'),
  ('sharpe', '0.7162'),
  ('sortino', '0.9624'),
  ('avg_drawdown', '-11.58'),
  ('time_in_drawdown', '0.9178'),
  ('calmar', '0.2313'),
  ('avg_return_to_drawdown', '1.367'),
  ('avg_loss', '-1.03'),
  ('avg_gain', '1.009'),
  ('gaintolossratio', '0.9795'),
  ('profitfactor', '1.132'),
  ('hitrate', '0.5362'),
  ('t_stat', '4.662'),
  ('p_value', '3.165e-06')],
 ('You can also plot / print:',
  ['rolling_ann_std', 'drawdown', 'curve', 'percent'])]

In [52]:
system.accounts.portfolio().percent.worst_drawdown()
system.accounts.portfolio().percent.quant_ratio_upper()
system.accounts.portfolio().percent.quant_ratio_lower()


1.5460082951566385

In [ ]:
system.accounts.methods()


['average_forecast',
 'capital_multiplier',
 'config',
 'forecast_cap',
 'forecast_diversification_multiplier',
 'forecast_turnover',
 'forecast_weight',
 'forecast_weights_for_instrument',
 'get_SR_cost_for_instrument_forecast',
 'get_SR_cost_given_turnover',
 'get_SR_cost_per_trade_for_instrument',
 'get_SR_holding_cost_only',
 'get_SR_trading_cost_only_given_turnover',
 'get_SR_transaction_cost_for_instrument_forecast',
 'get_actual_buffers_for_position',
 'get_actual_capital',
 'get_actual_position',
 'get_annual_risk_target',
 'get_average_position_at_subsystem_level',
 'get_average_position_for_instrument_at_portfolio_level',
 'get_buffered_position',
 'get_buffered_position_with_multiplier',
 'get_buffered_subsystem_position',
 'get_buffers_for_position',
 'get_buffers_for_subsystem_position',
 'get_capped_forecast',
 'get_daily_percentage_volatility',
 'get_daily_prices',
 'get_daily_returns_volatility',
 'get_fx_rate',
 'get_hourly_prices',
 'get_instrument_diversification_mul

In [57]:
system.accounts.pandl_for_instrument("SP500").percent.stats()


2026-05-27 12:47:56 DEBUG base_system {'stage': 'accounts', 'instrument_code': 'SP500'} Calculating pandl for instrument for SP500
2026-05-27 12:47:56 DEBUG base_system {'stage': 'accounts', 'instrument_code': 'SP500'} Calculating pandl for instrument for SP500


[[('min', '-21'),
  ('max', '5.805'),
  ('median', '0.042'),
  ('mean', '0.03322'),
  ('std', '1.021'),
  ('skew', '-1.362'),
  ('ann_mean', '8.504'),
  ('ann_std', '16.34'),
  ('sharpe', '0.5205'),
  ('sortino', '0.6494'),
  ('avg_drawdown', '-12.42'),
  ('time_in_drawdown', '0.938'),
  ('calmar', '0.1368'),
  ('avg_return_to_drawdown', '0.6849'),
  ('avg_loss', '-0.7689'),
  ('avg_gain', '0.7293'),
  ('gaintolossratio', '0.9485'),
  ('profitfactor', '1.097'),
  ('hitrate', '0.5362'),
  ('t_stat', '3.387'),
  ('p_value', '0.0007096')],
 ('You can also plot / print:',
  ['rolling_ann_std', 'drawdown', 'curve', 'percent'])]

In [71]:
system.rawdata.get_daily_percentage_returns("SP500")


2026-05-26 16:35:07 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-05-26 16:35:07 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-05-26 16:35:07 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-05-26 16:35:08 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'SP500'} Calculating daily prices for SP500


index
1982-09-14         NaN
1982-09-15    0.004026
1982-09-16   -0.005261
1982-09-17   -0.007338
1982-09-20    0.006480
                ...   
2024-03-22   -0.003356
2024-03-25   -0.001704
2024-03-26   -0.001565
2024-03-27    0.006735
2024-03-28   -0.000754
Freq: B, Length: 10838, dtype: float64

In [ ]:
system.rawdata.get_daily_percentage_returns("SP500").loc["2022-09-09"]


0.01401468788249694

In [ ]:
system.data.methods()


['all_asset_classes',
 'all_instruments_in_asset_class',
 'asset_class_for_instrument',
 'daily_prices',
 'db_futures_adjusted_prices_data',
 'db_futures_instrument_data',
 'db_futures_multiple_prices_data',
 'db_fx_prices_data',
 'db_roll_parameters',
 'db_spread_cost_data',
 'get_all_instrument_data_as_df',
 'get_backadjusted_futures_price',
 'get_current_and_forward_price_data',
 'get_fx_for_instrument',
 'get_instrument_asset_classes',
 'get_instrument_currency',
 'get_instrument_list',
 'get_instrument_meta_data',
 'get_instrument_object_with_meta_data',
 'get_instrument_raw_carry_data',
 'get_multiple_prices',
 'get_multiple_prices_from_start_date',
 'get_raw_cost_data',
 'get_raw_price',
 'get_raw_price_from_start_date',
 'get_roll_parameters',
 'get_rolls_per_year',
 'get_spread_cost',
 'get_value_of_block_price_move',
 'hourly_prices',
 'keys',
 'length_of_history_in_days_for_instrument',
 'methods',
 'start_date_for_data',
 'system_init']

In [51]:
system.data.get_raw_price("SP500")


index
1982-09-14     405.40
1982-09-15     405.90
1982-09-16     405.25
1982-09-17     404.35
1982-09-20     405.15
               ...   
2022-09-26    3667.25
2022-09-27    3665.25
2022-09-28    3729.75
2022-09-29    3658.75
2022-09-30    3641.50
Name: price, Length: 10188, dtype: float64

In [ ]:
system.stage_names


['accounts',
 'portfolio',
 'positionSize',
 'rawdata',
 'combForecast',
 'forecastScaleCap',
 'rules']

In [ ]:
from systems.provided.futures_chapter15.basesystem import *

config = (
    Config()
)  # using a default config so we know we have all instruments there in principle
system = futures_system(config=config)
instruments_with_adj_prices = system.data.get_instrument_list()
"SP500" in instruments_with_adj_prices


Private configuration '/Users/jasonli/Dev/FORKed repo/pysystemtrade/private/private_config.yaml' is missing or misconfigured; no problem if running in sim mode


True

In [ ]:
system.get_list_of_instruments_to_remove()
system.get_list_of_markets_with_trading_restrictions()
system.config.exclude_instrument_lists["trading_restrictions"]
system.get_list_of_bad_markets()
system.config.exclude_instrument_lists["bad_markets"]


2026-05-27 21:33:03 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-05-27 21:33:03 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-05-27 21:33:03 DEBUG base_system Following instruments have restricted trading:  ['RESTRICTED_EXAMPLE'] 
2026-05-27 21:33:03 DEBUG base_system Following instruments are marked as 'bad_markets':  ['BAD_EXAMPLE']


['BAD_EXAMPLE']

In [ ]:
system.get_list_of_duplicate_instruments_to_remove()
system.config.duplicate_instruments["exclude"]
system.config.duplicate_instruments["include"]
system.config.exclude_instrument_lists["ignore_instruments"]
system.get_list_of_bad_markets()


2026-05-27 15:40:13 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-05-27 15:40:13 DEBUG base_system Following instruments are marked as 'bad_markets':  ['BAD_EXAMPLE']


['BAD_EXAMPLE']

In [ ]:
system.get_list_of_bad_markets()


2026-05-27 21:07:58 DEBUG base_system Following instruments are marked as 'bad_markets':  ['BAD_EXAMPLE']


['BAD_EXAMPLE']

In [ ]:
system = futures_system()


Private configuration '/Users/jasonli/Dev/FORKed repo/pysystemtrade/private/private_config.yaml' is missing or misconfigured; no problem if running in sim mode


In [ ]:
rules = system.rules.trading_rules()
rules.keys()


dict_keys(['ewmac2_8', 'ewmac4_16', 'ewmac8_32', 'ewmac16_64', 'ewmac32_128', 'ewmac64_256', 'carry'])

In [ ]:
rules["ewmac8_32"].data


['rawdata.get_daily_prices', 'rawdata.daily_returns_volatility']

In [91]:
from systems.provided.rules.ewmac import ewmac_forecast_with_defaults as ewmac
from systems.forecasting import Rules
from systems.trading_rules import TradingRule

ewmac_rule = TradingRule(ewmac)
my_rules = Rules(dict(ewmac=ewmac_rule))
ewmac_rule


TradingRule; function: <function ewmac_forecast_with_defaults at 0x13c494790>, data: data.daily_prices (args: {}) and other_args: 

In [ ]:
from systems.provided.futures_chapter15.basesystem import futures_system

system = futures_system()

price = system.rawdata.get_daily_prices("SP500")


Configuring sim logging
2026-07-14 14:17:06 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'SP500'} Calculating daily prices for SP500


In [3]:
raw = system.rules.get_raw_forecast("SP500", "ewmac8_32")
capped = system.forecastScaleCap.get_capped_forecast("SP500", "ewmac8_32")
combined = system.combForecast.get_combined_forecast("SP500")


In [ ]:
position = system.portfolio.get_notional_position


In [ ]:
position("SP500")


2026-07-14 14:19:05 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'SP500'} Calculating notional position for SP500


KeyError: 'SP500'